## Aim:
How to link the rule-based extracted CI_TYPE-GEO pairs with the respective Ci failure impacts

**Idea**\
Test using prompt engineering by passing table of CIGEO pairs to GPT-J model.
Steps:
* Load model and apply it always on one chunk of the document to extract CI failure impacts 
* Use prompt engineering to extract time and location of the CI failure (origin) the CI impacts (impact location)
or 
* Pass dataframe of pairs as input to the model
or
* Use few shot prompting with example answers

**Finally:**
* Compaire all approaches of spatial and temporal linking CI failure impacts

In [1]:
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=0  # nvidia gpu
%env PYTORCH_ALLOC_CONF=expandable_segments:True
# %env TORCH_CUDA_ARCH_LIST=8.6

# settings for distributed computing
%env WORLD_SIZE=1
%env RANK=0
%env LOCAL_RANK=0

# NOTE: # WORLD_SIZE: each GPU corresponds to one process (world = no. of processes within a group), processes communicate with each other enabling eg., distributed training
# NOTE: # RANK: IDs of the processes, ranging from 0 up to WORLD_SIZE - 1

env: CUDA_DEVICE_ORDER=PCI_BUS_ID
env: CUDA_VISIBLE_DEVICES=0  # nvidia gpu
env: PYTORCH_ALLOC_CONF=expandable_segments:True
env: WORLD_SIZE=1
env: RANK=0
env: LOCAL_RANK=0


In [2]:
import os
import sys
import re
import glob
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
from jinja2 import Template
import spacy
import langextract as lx
import textwrap
from langchain_docling import DoclingLoader
from huggingface_hub import login
import torch




sys.path.append("../")
import src.settings as s

torch.manual_seed(42)

# set default location to store model before loading transformers
os.environ["HF_HOME"] = (
    "/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/"
)

/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate CI_GEO-pairs

In [ ]:
# import spacy_transformers


try:
    nlp = spacy.load(s.settings.SPACY_MODEL)
except (OSError, ValueError) as e:
    print(f"spaCy language model '{s.settings.SPACY_MODEL}' not found. Downloading ...")
    ## loading transformer language model for NER requires additional package
    if (s.settings.SPACY_MODEL.endswith("_trf") and importlib.util.find_spec("spacy[transformers]") is None):
        !uv add spacy[transformers]
    !uv run python -m spacy download {s.settings.SPACY_MODEL}
    nlp = spacy.load(s.settings.SPACY_MODEL)

print(f"Loaded spaCy language model: {s.settings.SPACY_MODEL}")

Loaded spaCy language model: en_core_web_trf


In [14]:
## Create New entity for transport infrastructure and apply it on any doc

## see for more info: https://spacy.io/usage/rule-based-matching#entityruler
## NOTE EntityRuler is hidden inside .add_pipe()


## call nlp model and create pipeline with new entity pattern
# NOTE Creating new entity (CI_TYPE) solves the issue that FAC entities (buildings, airports, highways, bridges, etc.) refer only to the name of the facility (e.g. A76, Ahrtalbahn)
config = {"spans_key": None, "annotate_ents": True, "overwrite": False}
try:
    ruler = nlp.add_pipe("span_ruler", config=config)
    ruler.from_disk("../ner_patterns.jsonl")
except ValueError:
    print("SpanRuler already exists in pipeline.")
    ruler = nlp.get_pipe("span_ruler")
    ruler.from_disk("../ner_patterns.jsonl")


# store patterns in jsonl file, example:
# ruler.add_patterns([
#     {"label": "CI_TYPE", "pattern": "road?.+"},
#    {"label":"CI_TYPE","pattern":"rail.*$"},
# ])
# ruler.to_disk("../ner_patterns.jsonl")


## load doc
PARSED_TEXT_DIR = "../" + s.settings.PATH_DATA + "parsed_documents/"
FILE_PATH = PARSED_TEXT_DIR + "Koks et al 2022 Brief communication_cleaned.md"
loader = DoclingLoader(FILE_PATH)  # use chunks from Docling.Loader
doc = loader.load()

## TODO make as pydantic class with fixed attributes
df_ci_geo = pd.DataFrame(
    columns=[
        "chunk_id",
        "ci_entity",
        "ci_entity_label",
        "geo_entity",
        "geo_entity_label",
        "token_distance",
    ]
)


## get most likely geolocation for each CI entity based on distance
for i, chunk in enumerate(doc):
    nlp_chunk = nlp(chunk.page_content)
    all_ents = [ent for ent in nlp_chunk.ents]
    ci_type_ents = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE", "FAC"]]
    ci_type_ents_info = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE"]]
    fac_ents_info = [ent for ent in nlp_chunk.ents if ent.label_ in ["FAC"]]

    # check if chunk contains CI_TYPE entities
    if len(ci_type_ents) > 0:
        print(f"\nChunk [{i}], No. CI_TYPE and FAC entities: {len(ci_type_ents)}")
        print(
            f"Contains following entities for CI_TYPE: {ci_type_ents_info}, FAC: {fac_ents_info}"
        )
        print(f"Chunk text [{i}]:", chunk.page_content)
        # print(f"{ {(ci_type_ents[i].text, ci_type_ents[i].label_) for i in range(len(ci_type_ents))} } ")

        # iterate over all entities within chunk
        for ent_idx in range(len(all_ents)):
            # when entity is CI_TYPE or FAC (i.e. buidling, airports, highways) do following ...
            if all_ents[ent_idx].label_ in ["CI_TYPE", "FAC"]:
                ci_idx = ent_idx

                ## .. calculate distances between CI_TYPE entity and  all GEO entities in chunk based on index position
                distance_list = []
                idx_in_chunk = []
                try:
                    for ent_idx in range(len(all_ents)):
                        # TODO calc distances between CI_TYPE ~ GEO entities based on word numbers and not entities
                        if all_ents[ent_idx].label_ in ["GPE", "LOC"]:
                            geo_idx = ent_idx
                            dist_ent_pair = np.abs(ci_idx - geo_idx)
                            distance_list.append(dist_ent_pair)
                            idx_in_chunk.append((ent_idx))
                            closest_pair_idx = np.argmin(
                                distance_list
                            )  # idx of closest GEO entity
                            distance_closest_pair = distance_list[closest_pair_idx]

                    threshold = 5  # max token distance between CI_TYPE and GEO entity
                    if distance_closest_pair > threshold:
                        print(
                            f""" Token distance between CI_TYPE/FAR and next GEO entity is {distance_closest_pair} and thus larger than the allowed distance of {threshold} tokens """
                        )
                        continue
                    else:
                        print(
                            f"""  Closest GEO entity to CI_TYPE/FAC entity "{all_ents[ci_idx]}" is "{all_ents[idx_in_chunk[closest_pair_idx]]}" at distance {distance_closest_pair}"""
                        )  # TODO constrain min.distance to max value (eg. 5 tokens), issue: likely when distance value is high that geolocation of Ci_type is mentioned in previous sentences or chunk

                    ## write as dict entry incl chunk_id, ci_entity, geo_entity, distance
                    result_dict = {
                        "chunk_id": i,
                        "ci_entity": all_ents[ci_idx].text,
                        "ci_entity_label": all_ents[ci_idx].label_,
                        "geo_entity": all_ents[idx_in_chunk[closest_pair_idx]].text,
                        "geo_entity_label": all_ents[
                            idx_in_chunk[closest_pair_idx]
                        ].label_,
                        "token_distance": distance_closest_pair,
                    }
                    df_ci_geo = pd.concat(
                        [df_ci_geo, pd.DataFrame([result_dict])], ignore_index=True
                    )

                except IndexError:
                    print("No GEO entities found in this chunk.")
                    continue
                # print("\nidx_in_chunk, closest pair idx", idx_in_chunk, closest_pair_idx)

                # spacy.displacy.render(
                #     nlp_chunk, style="ent",
                #     options={"ents": ["CI_TYPE", "GPE", "LOC", "FAC"], "colors": {"CI_TYPE": "violet"}}
                # )

        else:
            print("\nNo CI_TYPE or FAC entities found in this chunk.")
            continue

SpanRuler already exists in pipeline.


2025-12-04 23:00:54,839 - INFO - Going to convert document batch...
2025-12-04 23:00:54,840 - INFO - Initializing pipeline for SimplePipeline with options hash 4cc01982ae99b46a2a63fcda46c47c35
2025-12-04 23:00:54,840 - INFO - Processing document Koks et al 2022 Brief communication_cleaned.md
2025-12-04 23:00:54,880 - INFO - Finished converting document Koks et al 2022 Brief communication_cleaned.md in 0.04 sec.
Token indices sequence length is longer than the specified maximum sequence length for this model (556 > 512). Running this sequence through the model will result in indexing errors



Chunk [1], No. CI_TYPE and FAC entities: 3
Contains following entities for CI_TYPE: [bridges, schools, hospitals], FAC: []
Chunk text [1]: Abstract. Germany, Belgium and the Netherlands were hit by extreme precipitation and ﬂooding in July 2021. This brief communication provides an overview of the impacts to large-scale critical infrastructure systems and how recovery has progressed. The results show that Germany and Belgium were particularly affected, with many infrastructure assets severely damaged or completely destroyed. Impacts range from completely destroyed bridges and sewage systems, to severely damaged schools and hospitals. We ﬁnd that (large-scale) risk assessments, often focused on larger (river) ﬂood events, do not ﬁnd these local, but severe, impacts due to critical infrastructure failures. This may be the result of limited availability of validation material. As such, this brief communication not only will help to better understand how critical infrastructure can be aff

In [15]:
df_ci_geo.loc[df_ci_geo["ci_entity"] == "rail"]  # .head(15)


,chunk_id,ci_entity,ci_entity_label,geo_entity,geo_entity_label,token_distance
17,6,rail,CI_TYPE,the Ahr valley,LOC,2
46,11,rail,CI_TYPE,Altenburg,GPE,4


In [17]:
doc[13].page_content

'We found no information regarding direct impact on solid-waste facilities as a result of the ﬂood event. However, there is a large pressure on the solid-waste sector to clean the affected areas; 1 month after the event, we observed dozens of large temporary waste ﬁlls and frequent incidences of oil pollution in Rhineland-Palatinate during a ﬁeld visit. In the Ahrweiler district alone, the ﬂood caused as much solid waste as normally would be collected over 30 years. In Belgium, the amount of solid waste is estimated around 160 000 t, stored at several places, such as the abandoned highway track A601. This highway has been used for approximately 9 months as a temporary storage for debris (Couplez, 2022). In the Netherlands, there have been primarily problems with waste deposits along the river banks, which is mostly the solid waste transported by the river from further upstream. Thousands of tonnes of tree debris (logs and\nNat. Hazards Earth Syst. Sci., 22, 3831–3838, 2022\nE. E. Koks 

## LLama with LangExtract

###  Prompt engineering

In [6]:

question = "Which impacts of infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, societal or economic impacts, the location and possibly the time of the infrastructure failure."

## without s+e impacts
# question = "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location and possibly the time of the infrastructure failure."

In [7]:
# TODO move template to separate file and load via get_template(), define conditions (e.g. user is technical or not)
# TODo make pydantic class model for expected JSON output

# Example code: https://medium.com/@alecgg27895/jinja2-prompting-a-guide-on-using-jinja2-templates-for-prompt-management-in-genai-applications-e36e5c1243cf
# test instead of user_type (see: {% block user_type %}) the modification of question in regard to CI impact types (Tier 1,2,3 and 4 )


## first
#     You are an expert analyst assistant and should use ONLY the provided context to answer the following question:

## Text_snippet column
#     In the field "text_snippet" provide the exact linenumbers (within a list) from the context which you used to extract the information about the infrastructure failure and its impacts.
## too long responses


# prompt_template = """

#     You are an expert analyst assistant and should use ONLY the provided context to answer the following question:
    
#     Question: "{{ question }}"
       
#     Context:
#     {% for item in context %}
#     - {{ item.text }} (Citation: {{ item.citation }})
#     {% endfor %}

#     Evaluate and improve your answer based on the information about critical infrastructure (CI) types (column: "ci_entity") and their geolocations (column: "geo_entity") mentioned in CI locations.
    
#     CI locations:
#     {% for item in context %}
#     - ("ci_entity" and "geo_entity":\n {{ item["ci_locations"][["ci_entity", "geo_entity"]] }})
#     {% endfor %}


#     In the field "confidence" you should give an estimate how confident you are about the provided information on a scale between 1 (low) to 5 (high). 
#     In the "confidence_explanation" field give also a short explanation why you decided for a certain confidence value (maximum two bullet points within a list).
    

#     Return ONLY valid JSON in the following list format:
#     [{
#         "infrastructure_type": "...",
#         "damage": "...",
#         "location": "...",
#         "time": "...",
#         "duration": "...",
#         "confidence": "...",
#         "confidence_explanation": "[...]"
#     }]

#     Each nested dictionary describes one failure case.
#     DO NOT add commentary or text outside the JSON.
    

#     Answer:
# """

######################################################################


prompt_template = """

    You are an expert analyst assistant and should use ONLY the provided context to answer the following question:
    
    Question: "{{ question }}"
       
    Context:
    {% for item in context %}
    - {{ item.text }} (Citation: {{ item.citation }})
    {% endfor %}

    
    For the the fields "societal_impact" and "economic_impact" you should try to extract information about societal or economic consequences of infrastructure failures mentioned in the context.
    However, if you do not find any information about societal or economic consequences, then return for these fields a "NAN" value.
    
    For the fields ""infrastructure_type" and "location" you should evaluate and improve your answer based on the information mentioned in CI locations.

    CI locations:
    {% for item in context %}
    - ("ci_entity" and "geo_entity":\n {{ item["ci_locations"][["ci_entity", "geo_entity"]] }})
    {% endfor %}


    In the field "confidence" you should give an estimate how confident you are about the provided information in regard to the societal and economic impacts on a scale between 1 (low) to 5 (high). 
    In the "confidence_explanation" field give also a very short explanation why you decided for a certain confidence value (maximum two bullet points within a list).


    Return ONLY valid JSON in the following list format:
    [{
        "infrastructure_type": "...",
        "damage": "...",
        "societal_impact": "...",
        "economic_impact": "...",
        "location": "...",
        "confidence_explanation": "[...]",
        "confidence": "...",
    }]

    Each nested dictionary describes one failure case.
    DO NOT add commentary or text outside the JSON.
    

    Answer:
"""

template = Template(prompt_template)


# example_context = doc[6].page_content
# chunk_id = 6
# context = [
#     {
#         "text": example_context,
#         "citation": "Koks et al., 2022",
#         "ci_locations": df_ci_geo.loc[df_ci_geo["chunk_id"]==chunk_id],#to_dict(orient="records")
#     },  # TODO use author names or Primary keys from DB
#     # {"text": context, "citation": "Meier et al., 2025"},
# ]

# rendered_prompt = template.render(
#     context=context,
#     question=question,
#     # messages=messages
# )
# print(rendered_prompt)


## left overs
#  Try to be as specific as possible in your answer (bullet points), mention the impacts as numerical information along the location of the impact, and refer to the citations provided in the context.
# # Extract information about infrastructure failures based on the following question:

##  Test Gemini with LangExtract
Langextract does not support Meta models (eg llama) directly (only via Ollama). For this reason we use an alternative for now with an easier implementation: Googles gemini\

**Local models with Ollama**\
Later we can replace Gemini with Mistral, Claude or an local Ollama model eg. via [`result_local = lx.extract(..., model_id="ollama:llama2")`](https://wandb.ai/wandb_fc/genai-research/reports/LangExtract-Transform-text-into-structured-data-with-AI--VmlldzoxNDI1OTMyNw#:~:text=LangExtract%20is%20open%2Dsource%20and,without%20requiring%20any%20fine%2Dtuning.
) 
```
# test with local model from Ollama
result = lx.extract(
    text_or_documents=input_text,
    prompt_description=prompt,
    examples=examples,
    model_id="gemma2:2b",  # Automatically select Ollama provider
    model_url="http://localhost:8002",  # check if needed 
    fence_output=False,
    use_schema_constraints=False
)
```
Optional: if you use API-based packages eg. fastchat and vLLM for passing HF models to LangExtract. Keep in mind that these packages are needed as LangExtract requires the llm (when from HF, or Llama-2) in an api-like structure (here port: 8001)
See usage examples, https://github.com/google/langextract?utm_source=chatgpt.com

### Basic usage test of LangExtract
Example taken from langExtract github 
set api key for Gemini in `.env`

In [ ]:
# 1. Define the prompt and extraction rules
prompt = textwrap.dedent("""\
    Extract characters, emotions, and relationships in order of appearance.
    Use exact text for extractions. Do not paraphrase or overlap entities.
    Provide meaningful attributes for each entity to add context.""")

# 2. Provide a high-quality example to guide the model
examples = [
    lx.data.ExampleData(
        text="ROMEO. But soft! What light through yonder window breaks? It is the east, and Juliet is the sun.",
        extractions=[
            lx.data.Extraction(
                extraction_class="character",
                extraction_text="ROMEO",
                attributes={"emotional_state": "wonder"}
            ),
            lx.data.Extraction(
                extraction_class="emotion",
                extraction_text="But soft!",
                attributes={"feeling": "gentle awe"}
            ),
            lx.data.Extraction(
                extraction_class="relationship",
                extraction_text="Juliet is the sun",
                attributes={"type": "metaphor"}
            ),
        ]
    )
]


In [ ]:
# The input text to be processed
input_text = "Lady Juliet gazed longingly at the stars, her heart aching for Romeo"

# Run the extraction
result = lx.extract(
    text_or_documents=input_text,
    prompt_description=prompt,
    examples=examples,
    model_id="gemini-2.5-flash",
)

LangExtract: model=gemini-2.5-flash, current=68 chars, processed=0 chars:  [00:02]


In [ ]:
result

AnnotatedDocument(extractions=[Extraction(extraction_class='character', extraction_text='Lady Juliet', char_interval=CharInterval(start_pos=0, end_pos=11), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=1, group_index=0, description=None, attributes={'emotional_state': 'longing'}), Extraction(extraction_class='emotion', extraction_text='aching', char_interval=CharInterval(start_pos=52, end_pos=58), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=2, group_index=1, description=None, attributes={'feeling': 'painful longing'}), Extraction(extraction_class='relationship', extraction_text='for Romeo', char_interval=CharInterval(start_pos=59, end_pos=68), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=3, group_index=2, description=None, attributes={'type': 'romantic longing'})], text='Lady Juliet gazed longingly at the stars, her heart aching for Romeo')

In [ ]:
lx.io.save_annotated_documents([result], output_name="extraction_results.jsonl", output_dir=".")

# Generate the visualization from the file
html_content = lx.visualize("extraction_results.jsonl")
with open("visualization.html", "w") as f:
    if hasattr(html_content, 'data'):
        f.write(html_content.data)  # For Jupyter/Colab
    else:
        f.write(html_content)

LangExtract: Saving to extraction_results.jsonl: 1 docs [00:00, 119.41 docs/s]

✓ Saved 1 documents to extraction_results.jsonl



LangExtract: Loading extraction_results.jsonl: 100%|██████████| 909/909 [00:00<00:00, 3.40MB/s]

✓ Loaded 1 documents from extraction_results.jsonl


### Prompt and few-shot examples

In [4]:

# 1. Define the prompt and extraction rules
prompt = textwrap.dedent("""\

    Extract from the ocntext information about the flood-affected infrastructure_type, its damage, its location, as well as about 
    the societal or economic impacts which resulted from the infrastructure failure.
    However, if you do not find any information about societal or economic impacts, then return for these fields a "NAN" value.

    Use exact text for extractions. DO NOT paraphrase or overlap entities.
    Provide meaningful attributes for each entity to add context.

 """)

# 2. Provide some high-quality examples to guide the model
few_shot_examples = [
    lx.data.ExampleData(
        text="More than 130 km of motorways were closed directly after the event, of which 50 km were still closed two months later, with an estimated repair cost of EUR100 million (Hauser, 2021). ",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="motorways",
                attributes={"time": "directly after the event"}
            ),
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR100 million",
                attributes={"type": "repair cost"}
            ),
        ]
    ),
    lx.data.ExampleData(
        text="Of the 112 bridges in the flooded 40 km of the Ahr valley (Rhineland-Palatinate), 62 bridges were destroyed, 13 were severely damaged and only 35 were in operation a month after the flood event (MDR, 2021).",
        extractions=[
            lx.data.Extraction(
                extraction_class="infrastructure_type",
                extraction_text="bridges",
                attributes={"damage type": "destroyed", "number": "62"}
            ),
           lx.data.Extraction(
                extraction_class="location",
                extraction_text="Ahr valley",
                attributes={"region": "Rhineland-Palatinate"}),

        ]
    ),
    lx.data.ExampleData(
        text=" In total, at least 220 casualties have been reported, with insured loss estimates of approximately EUR 150 million–EUR 250 million in the Netherlands (Verbond voor Verzekeraars, 2022), " \
        "EUR 2.2 billion in Belgium (Assuralia, 2022) and EUR 8.2 billion (GDV, 2022) in Germany. " \
        "The event caused major damages to residential and commercial structures and to many critical infrastructure (CI) assets. ",
        extractions=[
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR 150 million–EUR 250 million",
                attributes={"type": "insured loss estimates", "location": "Netherlands", "citation": "Verbond voor Verzekeraars, 2022"}
            ),
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR 2.2 billion",
                attributes={"type": "insured loss estimates", "location": "Belgium", "citation": "Assuralia, 2022"}
            ),
            lx.data.Extraction(
                extraction_class="economic_impact",
                extraction_text="EUR 8.2 billion",
                attributes={"type": "insured loss estimates", "location": "Germany", "citation": "GDV, 2022"}
            ),
        ]
    ),
]



### Appy LangExtract

In [ ]:
PARSED_TEXT_DIR = "../" + s.settings.PATH_DATA + "parsed_documents/"


filename = "Koks et al 2022 Brief communication_cleaned.md"
filepath = Path(PARSED_TEXT_DIR, filename)
filename_stem = filepath.stem


print(f"\n\n ######## -------- Processing document: {filepath.name} -------- ######## \n")

## extract authors, publication year and title 
citation_pattern = r"(.*?)(\d{4})(.*)" # split at first occurrence of year
try:
    authors, year, title = re.findall(citation_pattern, filename_stem)[0]
    citation = f"{authors} {year}"
except AttributeError as e:
    print(f"Could not extract citation from title: {e}")
    citation = filename_stem


## load doc
with open(filepath, 'r') as file:
    content = file.read()
doc = [lx.data.Document(content)]  # wrap content in Document object


print(f"\n  #############  -------- Text-2-Data: {filepath.name} -------- #############  \n")


response = lx.extract(
    text_or_documents=doc[0].text,
    prompt_description=prompt,
    examples=few_shot_examples,
    model_id="gemini-2.5-flash", # "gemini-2.5-pro",
    extraction_passes=1, # decrease recall but process faster
    max_workers=4,
)
# response = lx.extract(
#     text_or_documents=doc[0].text,
#     prompt_description=prompt,
#     examples=few_shot_examples,
#     model_id="gpt-4o-mini",
#     fence_output=True,              # Required for OpenAI
#     use_schema_constraints=False    # OpenAI doesn't support constraints
# )
print(response)

lx.io.save_annotated_documents(
    [response], 
    output_name=f"langextract_{filename_stem}.jsonl", 
    output_dir=s.PATH_DATA + "/llm_outputs/"
)




 ######## -------- Processing document: Koks et al 2022 Brief communication_cleaned.md -------- ######## 


  #############  -------- Text-2-Data: Koks et al 2022 Brief communication_cleaned.md -------- #############  



LangExtract: model=gemini-2.5-pro, current=9,365 chars, processed=0 chars:  [00:00]2025-12-07 23:15:17,648 - INFO - AFC is enabled with max remote calls: 10.
2025-12-07 23:15:17,649 - INFO - AFC is enabled with max remote calls: 10.
2025-12-07 23:15:17,651 - INFO - AFC is enabled with max remote calls: 10.
2025-12-07 23:15:17,651 - INFO - AFC is enabled with max remote calls: 10.
2025-12-07 23:15:17,773 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-pro:generateContent "HTTP/1.1 429 Too Many Requests"
2025-12-07 23:15:17,775 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-pro:generateContent "HTTP/1.1 429 Too Many Requests"
2025-12-07 23:15:17,776 - INFO - AFC is enabled with max remote calls: 10.
2025-12-07 23:15:17,779 - INFO - AFC is enabled with max remote calls: 10.
2025-12-07 23:15:17,780 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-pro:

InferenceRuntimeError: Parallel inference error: Gemini API error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro\nPlease retry in 42.265357281s.', 'status': 'RESOURCE_EXHAUSTED', 'detai

In [8]:
response

# .io.save_annotated_documents([result], output_name="../../data/llm_outputs/extraction_results.jsonl", output_dir=".")

# # Generate the visualization from the file
# html_content = lx.visualize("../../data/llm_outputs/extraction_results.jsonl")
# with open("visualization.html", "w") as f:
#     if hasattr(html_content, 'data'):
#         f.write(html_content.data)  # For Jupyter/Colab
#     else:
#         f.write(html_content)


NameError: name 'response' is not defined

In [ ]:
lx.data.Document?

Init signature:
lx.data.Document(
    text: 'str',
    *,
    document_id: 'str | None' = None,
    additional_context: 'str | None' = None,
)
Docstring:     
Document class for annotating documents.

Attributes:
  text: Raw text representation for the document.
  document_id: Unique identifier for each document and is auto-generated if
    not set.
  additional_context: Additional context to supplement prompt instructions.
  tokenized_text: Tokenized text for the document, computed from `text`.
File:           ~/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/.venv/lib/python3.13/site-packages/langextract/core/data.py
Type:           type
Subclasses:     

In [39]:
doc[0].tokenized_text.__dir__()


['text',
 'tokens',
 '__module__',
 '__firstlineno__',
 '__annotations__',
 '__doc__',
 '__static_attributes__',
 '__dict__',
 '__weakref__',
 '__dataclass_params__',
 '__dataclass_fields__',
 '__replace__',
 '__hash__',
 '__init__',
 '__repr__',
 '__eq__',
 '__match_args__',
 '__new__',
 '__str__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__reduce__',
 '__getstate__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

In [2]:
import gc
import torch

# clean up after each document
gc.collect()
torch.cuda.empty_cache()

### Testing vLLM and FastChat

In [ ]:
# # install before langextract and vllm

# # --tensor-parallel-size 1 --gpu-memory-utilization 0.90 
# !python -m vllm.entrypoints.openai.api_server \
#     --model meta-llama/Llama-2-7b-chat-hf \
#     --dtype auto \
#     --port 8000 \
#     --gpu-memory-utilization 0.90 
        


## test with fastchat loading
# documentation: https://fastchat.mintlify.app/install
# uv add fschat[model_worker,webui]

import os
from huggingface_hub import login


login(
    token=os.environ["HUGGINGFACE_TOKEN"]
)  # TODO replace by using pydantic settings

## Import NOTE (cpu/cuda/OOM):
## for 7b we need around 20GB VRAM 
# first try to run in with some optimazation (--load-8bit) and to run with CUDA --> make sure to compile against local cuda by running in terminal (adapt cuda version):
# CUDACXX=/usr/local/cuda-13/bin/nvcc CMAKE_ARGS="-DLLAMA_CUBLAS=on -DCMAKE_CUDA_ARCHITECTURES=native" FORCE_CMAKE=1 pip install .  
# if GPU still results in OOM --> pin down to cpu only "--device cpu"
# start worker at port 8001, NOTE: set specific port only when default port s already used by other process (eg. docker container - vector db)
# !python3 -m fastchat.serve.controller --port 8004 & !python3 -m fastchat.serve.model_worker --model-path meta-llama/Llama-2-7b-chat-hf  --load-8bit --device cpu --port 8004

# start controller and model worker in temrinal
!python3 -m fastchat.serve.controller --port 8001  & python3 -m fastchat.serve.model_worker --model-path meta-llama/Llama-2-7b-chat-hf --device cpu --port 8005


# !python3 -m fastchat.serve.cli --model-path meta-llama/Llama-2-7b-chat-hf  --load-8bit

In [ ]:

import os
from vllm import LLM, SamplingParams
from huggingface_hub import login

login(token=os.environ["HUGGINGFACE_TOKEN"]) 


#### response

In [12]:
safety_df = df_responses.copy()

# save to disk along with prompt text


OUTPUT_DIR = "../" + s.settings.PATH_DATA + "llm_outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

outfile_name = "ci_failure_impacts_responses_llama2_with_eco_explicit_3.csv"
outfile_path = OUTPUT_DIR +  outfile_name


if not os.path.isfile(outfile_path):
    outfile_response_path = Path(outfile_path)
    outfile_prompt_path = Path(OUTPUT_DIR +  "prompt_" + outfile_name.replace(".csv", ".txt"))

    print(f"Saving prompt and LLm response to {outfile_response_path.parent} ...")
    with open(outfile_prompt_path, "w") as f:
        f.write(prompt_template)
    safety_df.to_csv(outfile_response_path, index=False)
else:
    print(f"Output file {Path(outfile_name).stem} already exists. Skip saving to avoid overwriting ...")



Saving prompt and LLm response to ../../data/llm_outputs ...


In [13]:
## 10 min


df_ci_geo

,chunk_id,ci_entity,ci_entity_label,geo_entity,geo_entity_label,token_distance
0,9,Hospital,CI_TYPE,Erftstadt,GPE,1


## Evaluation